In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_160.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_2104.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_1306.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_233.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_2771.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_1466.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_1410.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_2760.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_2844.jpg
/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset/Tomato_Leaf_Mold/img_2765.jpg
/kaggle/input/datasets

In [2]:
import os
import random
import shutil

source_root = "/kaggle/input/datasets/zunorain/tomato-dataset-v2/augmented_dataset"
target_root = "/kaggle/working/tomato"

splits = ["train", "val", "test"]

# create folders
for split in splits:
    os.makedirs(os.path.join(target_root, split), exist_ok=True)

for cls in os.listdir(source_root):
    class_path = os.path.join(source_root, cls)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)
    random.shuffle(images)

    n = len(images)

    train_end = int(0.8 * n)
    val_end = int(0.9 * n)

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    for split, img_list in zip(
        ["train", "val", "test"],
        [train_imgs, val_imgs, test_imgs]
    ):
        split_folder = os.path.join(target_root, split, cls)
        os.makedirs(split_folder, exist_ok=True)

        for img in img_list:
            src = os.path.join(class_path, img)
            dst = os.path.join(split_folder, img)

            try:
                shutil.copy(src, dst)
            except:
                pass

print("Dataset split complete")

Dataset split complete


load dataset

In [3]:
import tensorflow as tf
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds=tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/tomato/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
test_ds=tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/tomato/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
val_ds=tf.keras.utils.image_dataset_from_directory(
    "/kaggle/working/tomato/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
class_names=train_ds.class_names;
print(class_names)

2026-03-18 02:04:46.491463: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773799486.983406      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773799487.125701      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773799488.350512      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773799488.350559      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773799488.350563      24 computation_placer.cc:177] computation placer alr

Found 24483 files belonging to 10 classes.


I0000 00:00:1773799527.984292      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773799527.990383      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 3067 files belonging to 10 classes.
Found 3059 files belonging to 10 classes.
['Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [5]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [6]:
from tensorflow import keras

base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
inputs = keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [8]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
early_stop = keras.callbacks.EarlyStopping(
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10


I0000 00:00:1773799539.713106      79 cuda_dnn.cc:529] Loaded cuDNN version 91002


766/766 ━━━━━━━━━━━━━━━━━━━━ 56s 61ms/step - accuracy: 0.6730 - loss: 1.0030 - val_accuracy: 0.8434 - val_loss: 0.4493
Epoch 2/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 45s 59ms/step - accuracy: 0.8339 - loss: 0.4826 - val_accuracy: 0.8676 - val_loss: 0.4019
Epoch 3/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 46s 60ms/step - accuracy: 0.8568 - loss: 0.4223 - val_accuracy: 0.8758 - val_loss: 0.3804
Epoch 4/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 47s 61ms/step - accuracy: 0.8671 - loss: 0.3842 - val_accuracy: 0.8859 - val_loss: 0.3295
Epoch 5/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 47s 61ms/step - accuracy: 0.8726 - loss: 0.3608 - val_accuracy: 0.8951 - val_loss: 0.3047
Epoch 6/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 47s 61ms/step - accuracy: 0.8824 - loss: 0.3378 - val_accuracy: 0.8846 - val_loss: 0.3264
Epoch 7/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 47s 61ms/step - accuracy: 0.8843 - loss: 0.3301 - val_accuracy: 0.8889 - val_loss: 0.3206
Epoch 8/10
766/766 ━━━━━━━━━━━━━━━━━━━━ 47s 61ms/step - accuracy: 0.8924 - loss: 0.3100 - val_accurac

In [10]:
# unfreeze last layers only
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
766/766 ━━━━━━━━━━━━━━━━━━━━ 72s 81ms/step - accuracy: 0.7998 - loss: 0.6167 - val_accuracy: 0.8081 - val_loss: 0.8559
Epoch 2/5
766/766 ━━━━━━━━━━━━━━━━━━━━ 60s 79ms/step - accuracy: 0.8681 - loss: 0.3950 - val_accuracy: 0.8745 - val_loss: 0.4311
Epoch 3/5
766/766 ━━━━━━━━━━━━━━━━━━━━ 60s 78ms/step - accuracy: 0.8892 - loss: 0.3337 - val_accuracy: 0.8987 - val_loss: 0.3108
Epoch 4/5
766/766 ━━━━━━━━━━━━━━━━━━━━ 60s 78ms/step - accuracy: 0.9032 - loss: 0.2850 - val_accuracy: 0.9222 - val_loss: 0.2465
Epoch 5/5
766/766 ━━━━━━━━━━━━━━━━━━━━ 60s 79ms/step - accuracy: 0.9131 - loss: 0.2541 - val_accuracy: 0.9232 - val_loss: 0.2440


In [11]:
loss, acc = model.evaluate(test_ds)
print("Test Accuracy:", acc)

96/96 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.9185 - loss: 0.2456
Test Accuracy: 0.9214215874671936


In [12]:
model.save("/kaggle/working/tomato_model.keras")